# moe-sae discovery run (spec §20 step 4) — Colab Pro A100-40

All state lives in GCS; **this notebook is rerunnable** — if the session dies at any point, just run all cells again in a fresh session and the chain resumes where it stopped (capture checkpoints every 100 examples; training runs skip-if-complete).
All state lives in GCS; **this notebook is rerunnable** — if the session dies at any point, run all cells again and the chain resumes. Works from browser Colab **or the Google Colab VS Code extension** (attach via Select Kernel → Colab → A100; cells execute on the Colab VM either way).
Setup once: add a Colab secret named `GITHUB_TOKEN` (repo-read PAT) via the 🔑 panel, and set `BUCKET` below to a bucket you own (same project you'll bill GCS storage to, ~$15/mo for 770 GB).

In [ ]:
gpu = !nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
print(gpu)
assert any("A100" in g for g in gpu), "Runtime > Change runtime type > A100 GPU"

In [ ]:
import os, subprocess

os.environ["BUCKET"] = "gs://CHANGE-ME/discovery"  # <-- set your bucket

# --- GCS auth: browser-Colab popup if available, else gcloud device flow ---
try:
    from google.colab import auth
    auth.authenticate_user()
except Exception:
    if subprocess.run(["gcloud", "auth", "list", "--format=value(account)"],
                      capture_output=True, text=True).stdout.strip() == "":
        # VS Code extension path: paste the code from the printed URL
        !gcloud auth login --no-launch-browser --update-adc

# --- repo token: Colab secrets panel if available, else prompt/env ---------
token = os.environ.get("GITHUB_TOKEN")
if not token:
    try:
        from google.colab import userdata
        token = userdata.get("GITHUB_TOKEN")
    except Exception:
        from getpass import getpass
        token = getpass("GitHub PAT (repo read): ")

if not os.path.exists("moe-sae"):
    !git clone -q https://{token}@github.com/andrewbistras/moe-sae.git
%cd moe-sae
!git pull -q

In [ ]:
# Env: keep Colab's CUDA torch; pin transformers for the vendored model code.
!pip -q install -e . --no-deps
!pip -q install "transformers==4.38.2" timm einops sentencepiece safetensors datasets accelerate pyyaml
!python -c "import torch, moe_sae; print('torch', torch.__version__, 'cuda:', torch.cuda.is_available())"

In [ ]:
# The whole chain: capture (3 workers) → merge → stats → 14 runs → eval → gate.
# Rerunnable; watch logs/capture_w*.log for worker progress.
!bash scripts/cloud/colab_discovery.sh

In [ ]:
import json, pathlib
gate = pathlib.Path("reports/sanity_gate_discovery.json")
if gate.exists():
    print(json.dumps(json.loads(gate.read_text()), indent=2))
else:
    print("gate not reached yet — rerun the previous cell (it resumes)")